# NB-11 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-11**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-11.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This turns the representation architecture into a lineage audit: which source assertions, transformations, and artifacts support one retained result?

- [Volume V, Chapter 8 — From source records to research-ready state](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c8)
- [Volume V, Chapter 14 — Reproducibility and selective revalidation](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c14)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-11', require_key=True)
print(setup_status)


# NB-11 — One result and its full lineage
Companion to Volume V §§1–5, 8–10 and 14. Prerequisite: NB-15.
Trace one retained controlled result, distinguish a source key from its label,
and identify which outputs need review when a mapping changes.

This notebook reads the book's semantic prototype and a frozen API descriptor
inspection. It does not rewrite the registry, contact production during replay,
promote claims, or create synthetic observations.

In [ ]:
NOTEBOOK_ID='NB-11'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

inspection=inputs['inspection'];entry=inputs['requests']['geometry']
geometry=execute('geometry',entry['method'],entry['request'])
variability=geometry['diagnostics']['retained_scenario_variability']
card=copy.deepcopy(inspection['candidate_card']);card['relative_gap']=variability['relative_gram_gap']
result=dict(ratio=variability['relative_gram_gap'],gram=np.asarray(geometry['result']['gram']),mean_gram=np.asarray(variability['mean_gram']),gap=np.asarray(variability['gram_gap']),shape=inspection['shape'],ledger=inspection['ledger'],comparisons=inspection['comparisons'],candidate_card=card)
audit=inspection['registry_audit'];cap=inspection['capabilities']
print('Replayed NB-15 relative Gram gap:',result['ratio'])
print('Ordered node-by-path carrier H:',result['shape'])
print('Prototype status:',audit['fixture_status'])

## A displayed scalar is the end of a dependency chain
The statistic is ‖E[K]−K(E[E])‖F / ‖E[K]‖F, where the first term is the
expected scenario Gram and the second is the Gram of the mean exposure
operator. Here the two scenarios are retained intact/outage operators, with
the original declared 75/25 weights and identity branch metric.

We read T from the saved archive and use E=−T under NB-15's sign convention.
H uses sink-minus-source payout coordinates. No observed PJM price panel is
converted into a pretend physical operator. Input, method and notebook hashes
are checked against the earlier run manifest for this particular statistic.

In [ ]:
for item in result['ledger']:print(item)
fig,ax=plt.subplots(figsize=(12,3));ax.set_axis_off()
labels=['Frozen H, T\ninput archive','E = −T\nordered carriers','75/25 scenarios\nidentity metric','Expected Gram\nand mean Gram','Relative gap\n0.099568']
for i,label in enumerate(labels):
    ax.text(i,.5,label,ha='center',va='center',bbox=dict(boxstyle='round,pad=.65',facecolor='#3B4252',edgecolor='#88C0D0'),fontsize=10)
    if i<4:ax.annotate('',xy=(i+.72,.5),xytext=(i+.28,.5),arrowprops=dict(arrowstyle='->',color='#A3BE8C'))
ax.set(xlim=(-.55,4.55),ylim=(0,1),title='One reproducible result · lineage is not a causal claim');fig.tight_layout();plt.show()

## Source identifier, target, assertion, channel and representation
A source identifier is an equality key within its namespace, not a mutable
display name. The inspected PJM descriptor names pnode_id as the identifier
and explicitly rejects pnode_name as identity. It supplies price components,
not a physical grid. Its latest view is not a historical as-of view.

The European prototype exposes two different source series (N2EX and APX)
through two channels but maps both to one old GB day-ahead target. The revised
manuscript requires separate provider-specific Market Index Price targets and
a versioned selected-MIP benchmark. The known reprojection note is authoritative
about this local implementation gap; an old successful registry build does not
make the old semantics current.

In [ ]:
print('Descriptor inspection:',cap['captured_at'])
print('PJM contract:',cap['pjm'])
for channel in audit['channels']:print(channel)
print('Required local reprojection:',audit['required_reprojection'])
print('Generation-vintage API gate:',cap['generation'])
assert cap['pjm']['name_is_identity'] is False
assert audit['mapping_status']=='prototype_requires_reprojection'

## Compare two existing method configurations without overwriting history
NB-15 already compared the scenario expectation and mean-operator Gram.
These answer different questions; the difference is operator variability.
Both candidate run specifications below retain distinct method identities and
content hashes. The original result remains addressable. No new market sample
or method-performance claim is introduced.

In [ ]:
for config in result['comparisons']:print(config)
assert result['comparisons'][0]['config_sha256']!=result['comparisons'][1]['config_sha256']
from matplotlib.colors import LinearSegmentedColormap
nord=LinearSegmentedColormap.from_list('nord_gap',['#2E3440','#88C0D0','#ECEFF4'])
fig,ax=plt.subplots(figsize=(7,5));im=ax.imshow(np.abs(result['gap']),cmap=nord)
ax.set(xlabel='Retained path column',ylabel='Retained path column',title='Absolute scenario/mean Gram difference · 18 paths')
fig.colorbar(im,ax=ax,label='Squared exposure units');fig.tight_layout();plt.show()

## Selective revalidation, with explicit unaffected outputs
Start from the existing EUR-per-GBP field and traverse deterministic
transformation dependencies in the prototype. Include linked research objects
as review candidates, not automatic failures. A corrected FX mapping affects
normalized GB prices and dependent spreads; it does not change the primitive
provider histories or the separate wind-revision calculation.

This is a bounded dependency analysis of the retained prototype. It cannot
certify that every production consumer has been found. Several affected names
still use the prototype's obsolete GB day-ahead terminology and require review.

In [ ]:
entry=inputs['requests']['impact']
projected=execute('impact',entry['method'],entry['request'])['result']['lineage_view']
impact=projected['legacy_impact']
assert projected['historical_replay_supported'] is False
assert projected['claim_promoted'] is False

print('Changed field:',impact['changed'])
print('Dependent review candidates:')
for name in impact['dependent_candidates']:print(' ',name)
print('Unaffected outputs within this graph:')
for name in impact['unaffected_outputs_in_this_graph']:print(' ',name)
assert any('forecast.revision' in x for x in impact['unaffected_outputs_in_this_graph'])
fig,ax=plt.subplots(figsize=(9,3));ax.barh(['Dependent candidates','Unaffected output fields'],[len(impact['dependent_candidates']),len(impact['unaffected_outputs_in_this_graph'])],color=['#D08770','#A3BE8C'])
ax.set(xlabel='Count within the declared prototype graph',title='Review scope · different object categories, not a completeness score');fig.tight_layout();plt.show()

## Candidate knowledge card and capability gaps
The card remains candidate and unreviewed. It references the retained MethodRun
and evidence checksum; StudyRun registration is explicitly absent. This is a
local handoff artifact, not a guessed production Research Steward schema.

Open gaps: prototype reprojection; effective-dated real-market identity;
unsupported historical PJM as-of views; the generation-vintage catalog/descriptor
conflict; and an unverified production Steward adapter. None requires an
ontology rewrite before this controlled notebook can run.

In [ ]:
print(json.dumps(result['candidate_card'],indent=2))
assert result['candidate_card']['automatic_promotion'] is False

## Exercise
Which parts of this chain would need review if the path ordering changed?
Which would change if only a node's display name changed? Distinguish a
deterministic dependency from a physical identity or an economic mechanism.

In [ ]:
for control in inputs['negative_controls']:
    reject(control['method'],control['request'])
checked=dict(metric=result,impact=impact,registry_audit=audit,capabilities=cap,typed_lineage=projected,new_synthetic_data=False,scientific_acceptance=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,default=lambda x:x.tolist(),indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(retained_metric_reproduced=True,input_method_hashes_verified=True,prototype_gap_explicit=True,selective_revalidation=True,candidate_only_no_registry_write=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)